In [1]:
from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/KP/YASA_RUNNING DETECTION/v1.2.0"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/KP/YASA_RUNNING DETECTION/v1.2.0


In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 123.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 106.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu1

In [3]:
!pip install mediapipe

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-contrib-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 8.7 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: opencv-contrib-python
    Found existing installation: opencv-contrib-python 4.12.0.88
    Uninstalling opencv

In [2]:
import os
import cv2
import glob
import yaml
import random
import pathlib
import numpy as np
import pandas as pd
from PIL import Image
import mediapipe as mp

import seaborn as sns
from tqdm import tqdm
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from tqdm.notebook import trange, tqdm
from IPython.display import Image, Video
from IPython.display import Image, display
from sklearn.metrics import precision_score, recall_score
!wandb disabled
import warnings
warnings.filterwarnings('ignore')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
W&B disabled.


In [5]:
import cv2
from ultralytics import YOLO
from collections import defaultdict, deque
import numpy as np
import mediapipe as mp
from tqdm import tqdm
import os
import time


def detect_persons_with_pose(model, input_video_path, output_video_path,
                              hip_threshold=7, knee_threshold=15, history_len=20):
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5, model_complexity=2)

    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        print(f"[ERROR] Could not open video: {input_video_path}")
        pose.close()
        return None  # Return None to indicate failure

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    out = cv2.VideoWriter(output_video_path,
                          cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))

    pose_history = defaultdict(lambda: deque(maxlen=history_len))

    total_time = 0
    total_frames = 0

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        start_time = time.time()
        yolo_results = model.track(frame, classes=0, persist=True, verbose=False)
        annotated_frame = frame.copy()

        if yolo_results[0].boxes.id is not None:
            boxes = yolo_results[0].boxes.xyxy.cpu().numpy()
            track_ids = yolo_results[0].boxes.id.int().cpu().tolist()

            for box, track_id in zip(boxes, track_ids):
                x1, y1, x2, y2 = map(int, box)

                label = "Unknown"
                color = (255, 165, 0)
                display_text = f"ID {track_id}: {label}"

                person_crop = frame[max(0, y1):y2, max(0, x1):x2]
                if person_crop.size == 0:
                    continue

                image_rgb = cv2.cvtColor(person_crop, cv2.COLOR_BGR2RGB)
                pose_results = pose.process(image_rgb)

                if pose_results.pose_landmarks:
                    landmarks = np.array([[lm.x * person_crop.shape[1] + x1,
                                            lm.y * person_crop.shape[0] + y1]
                                           for lm in pose_results.pose_landmarks.landmark])
                    pose_history[track_id].append(landmarks)

                    for i, landmark in enumerate(landmarks):
                        if i in [mp_pose.PoseLandmark.LEFT_HIP.value,
                                 mp_pose.PoseLandmark.RIGHT_HIP.value,
                                 mp_pose.PoseLandmark.LEFT_KNEE.value,
                                 mp_pose.PoseLandmark.RIGHT_KNEE.value]:
                            px, py = int(landmark[0]), int(landmark[1])
                            cv2.circle(annotated_frame, (px, py), 4, (255, 255, 0), -1)

                    if len(pose_history[track_id]) == pose_history[track_id].maxlen:
                        history = np.array(pose_history[track_id])

                        left_hip_y = history[:, mp_pose.PoseLandmark.LEFT_HIP.value, 1]
                        right_hip_y = history[:, mp_pose.PoseLandmark.RIGHT_HIP.value, 1]
                        avg_hip_y = (left_hip_y + right_hip_y) / 2
                        hip_oscillation = np.std(avg_hip_y)

                        left_knee_y = history[:, mp_pose.PoseLandmark.LEFT_KNEE.value, 1]
                        right_knee_y = history[:, mp_pose.PoseLandmark.RIGHT_KNEE.value, 1]
                        avg_knee_y = (left_knee_y + right_knee_y) / 2
                        knee_lift_range = np.ptp(avg_knee_y)

                        if hip_oscillation > hip_threshold and knee_lift_range > knee_threshold:
                            label = "Running"
                            color = (0, 0, 255)
                        else:
                            label = "Walking"
                            color = (0, 255, 0)

                        display_text = f"ID {track_id}: {label} (Osc:{hip_oscillation:.1f}, Knee:{knee_lift_range:.1f})"

                cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(annotated_frame, display_text, (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        out.write(annotated_frame)
        total_time += time.time() - start_time
        total_frames += 1

    cap.release()
    out.release()
    pose.close()

    avg_time_per_frame = total_time / total_frames if total_frames > 0 else None
    return avg_time_per_frame

In [6]:
if __name__ == "__main__":
    model = YOLO('yolo11n.pt')

    for i in tqdm(range(1, 11), desc="Processing videos", unit="video"):
        input_video_path = f'sample_{i}.mp4'
        output_video_path = f'result_{i}.mp4'

        if not os.path.exists(input_video_path):
            tqdm.write(f"[WARNING] {input_video_path} not found, skipping...")
            continue

        avg_time = detect_persons_with_pose(model, input_video_path, output_video_path)

        if avg_time is not None:
            tqdm.write(f"[INFO] {input_video_path} processed. Avg time/frame: {avg_time*1000:.2f} ms")
        else:
            tqdm.write(f"[ERROR] Failed to process {input_video_path}")

Processing videos:  10%|█         | 1/10 [01:06<09:59, 66.62s/video]

[INFO] sample_1.mp4 processed. Avg time/frame: 703.25 ms


Processing videos:  20%|██        | 2/10 [04:01<17:24, 130.52s/video]

[INFO] sample_2.mp4 processed. Avg time/frame: 858.39 ms


Processing videos:  30%|███       | 3/10 [05:21<12:32, 107.50s/video]

...

[INFO] sample_8.mp4 processed. Avg time/frame: 471.81 ms


Processing videos:  90%|█████████ | 9/10 [12:54<01:05, 65.55s/video]

[INFO] sample_9.mp4 processed. Avg time/frame: 406.67 ms


Processing videos: 100%|██████████| 10/10 [13:35<00:00, 81.57s/video]

[INFO] sample_10.mp4 processed. Avg time/frame: 1002.84 ms
